In [14]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/faizanhaider01/scaled-attendence-score/ridge_dataset.csv


In [15]:
df = pd.read_csv('/kaggle/input/datasets/faizanhaider01/scaled-attendence-score/ridge_dataset.csv')

In [16]:
df.sample(4)

,study_hours,attendance_pct,score
97,5.4203,3.7266,18.5415
133,4.7009,2.9286,15.6371
114,7.0672,8.6878,43.4455
46,4.4937,5.4255,26.7207


In [17]:
X = df.iloc[:,0:2]
y = df.iloc[:,-1]

In [18]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=4)

In [19]:
from sklearn.linear_model import Ridge
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import r2_score

# Using SgdRegressor

In [20]:
reg = SGDRegressor(penalty='l2',max_iter=40,eta0=0.001,learning_rate='constant',alpha=0.001)

In [21]:
reg.fit(X_train,y_train)
y_pred = reg.predict(X_test)
print("R2 score",r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

R2 score 0.748976808625774
[2.98675269 1.97226654]
[0.3015615]


# Using Ridge class:

In [22]:
reg_ = Ridge(alpha=0.0001, max_iter=50,solver='sparse_cg')

In [23]:
reg_.fit(X_train,y_train)

y_pred = reg_.predict(X_test)
print("R2 score",r2_score(y_test,y_pred))
print(reg_.coef_)
print(reg_.intercept_)

R2 score 0.7446404629051743
[3.05630869 2.01051653]
-0.31500675002845924


In [24]:
class MeraRidgeGD:
    
    def __init__(self,epochs,learning_rate,alpha):
        
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.alpha = alpha
        self.coef_ = None
        self.intercept_ = None
        
    def fit(self, X_train, y_train):
    
        self.coef_ = np.ones(X_train.shape[1])
        self.intercept_ = 0
        thetha = np.insert(self.coef_, 0, self.intercept_)
    
        X_train = np.insert(X_train, 0, 1, axis=1)
    
        for i in range(self.epochs):
            thetha_der = np.dot(X_train.T, X_train).dot(thetha) - np.dot(X_train.T, y_train) + self.alpha * thetha
            thetha_der[0] -= self.alpha * thetha[0]   # remove L2 penalty from intercept
            thetha = thetha - self.learning_rate * thetha_der
    
        self.coef_ = thetha[1:]
        self.intercept_ = thetha[0]
    def predict(self,X_test):
        
        return np.dot(X_test,self.coef_) + self.intercept_

In [25]:
# Try these values instead
reg = MeraRidgeGD(epochs=50, alpha=0.0001, learning_rate=0.0001)

In [26]:
reg.fit(X_train,y_train)

y_pred = reg.predict(X_test)
print("R2 score",r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

R2 score 0.7475659069633638
[3.01190453 1.97476508]
0.20241679843741675
